Build the Neural Network
========================

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaust-vislab/introduction-to-data-science-workshop/blob/spring-2025/notebooks/05-Build-models.ipynb)

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


## Excercise 1 - Load all the necessary libraries

<details><summary><b>Solution</b></summary> <pre>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
``` 
</pre></details>

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.

When training a neural network, it is important to utilize the available hardware [accelerators](https://pytorch.org/docs/stable/torch.html#accelerators) to speed up the training process. PyTorch supports various accelerators such as :

1. CUDA (for NVIDIA GPUs), 
2. MPS (for Apple Silicon), 
3. MTIA (Meta Training and Inference Accelerator), and 
4. XPU (Intel's heterogeneous computing platform). 

If none of these accelerators are available, the training will fall back to using the CPU.

In these exercises, we will define the device to be used for training and then create an instance of `NeuralNetwork`, move it to the selected device, and print its structure.

## Exercise 2 - Define the device to be used

**Define the Device**: Check for the availability of MPS, CUDA, MTIA, and XPU in that order, and set the device accordingly. If none of these are available, default to using the CPU.





<details><summary><b>Solution</b></summary> <pre>

```python
# Check if MPS (Metal Performance Shaders) is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
# Check if CUDA is available
elif torch.cuda.is_available():
    device = torch.device("cuda")
# Check if MTIA is available (Meta Training and Inference Accelerator)
elif torch.mtia.is_available():
    device = torch.device("mtia")
# Check if XPU is available ((Intel's heterogeneous computing platform).)
elif torch.xpu.is_available():
    device = torch.device("xpu")
# Default to CPU if no accelerators are available
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

```
</pre></details>

Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.

```python
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() # use all the features and methods provided by nn.Module
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

```




## Exercise 3 - Define the device to be used

**Create and Move Model**: Create an instance of `NeuralNetwork`, move it to the selected device, and print its structure.

<details><summary><b>Solution</b></summary> <pre>

```python
model = NeuralNetwork().to(device)
print(model)
```
</pre></details>

To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.

# Making Predictions with the Model
===================================

Once we have trained our model, we can use it to make predictions on new data. In this exercise, we will create a random input tensor, pass it through the model to get the logits, apply the softmax function to get the predicted probabilities, and then determine the predicted class.

## Exercise 4 - Make a Prediction

1. **Create a Random Input Tensor**: Create a random input tensor `X` with the shape `(1, 28, 28)` and move it to the selected device.
2. **Get Logits**: Pass the input tensor through the model to get the logits.
3. **Apply Softmax**: Apply the softmax function to the logits to get the predicted probabilities.
4. **Determine Predicted Class**: Find the class with the highest probability and print the predicted class.


<details><summary><b>Solution</b></summary> <pre> 

```python
import torch 
import torch.nn as nn
# Assuming the device and model are already defined
# Create a random input tensor and move it to the selected device
X = torch.rand(1, 28, 28, device=device)

# Pass the input tensor through the model to get the logits
logits = model(X)

# Apply the softmax function to get the predicted probabilities
pred_probab = nn.Softmax(dim=1)(logits)

# Find the class with the highest probability
y_pred = pred_probab.argmax(1)

# Print the predicted class
print(f"Predicted class: {y_pred}") 

```
</pre></details>


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.

## Exercise 5 - Create an Input Image

Create a random input tensor `input_image` with the shape `(3, 28, 28)` and print its size.

<details><summary><b>Solution</b></summary> <pre> 

```python
import torch
input_image = torch.rand(3, 28, 28) 
print(input_image.size())
``` 

</pre></details>

nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).

## Exercise 6 - Flatten the Input Image
Use the `nn.Flatten` layer to flatten the `input_image` and print the size of the flattened image.

<details><summary><b>Solution</b></summary> <pre> 

```python
flatten = nn.Flatten() 
flat_image = flatten(input_image) 
print(flat_image.size()) 
```
</pre>
</details>

nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.

## Exercise 3 - Pass Through a Linear Layer
Create a linear layer with `in_features=28*28` and `out_features=20`. Pass the `flat_image` through this layer and print the size of the output.


<details><summary><b>Solution</b></summary> <pre>

```python 
layer1 = nn.Linear(in_features=28*28, out_features=20) hidden1 = layer1(flat_image) 
print(hidden1.size()) 
```
</pre>
</details>


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.

## Exercise 4 - Apply ReLU Activation
Print the output of the linear layer before and after applying the ReLU activation function.



<details><summary><b>Solution</b></summary> <pre> 

```python
print(f"Before ReLU: {hidden1}\n\n") 
hidden1 = nn.ReLU()(hidden1) 
print(f"After ReLU: {hidden1}") 
```

</pre> 
</details>

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.

## Exercise 5 - Create a Sequential Model
Create a sequential model that includes the flatten layer, the linear layer, a ReLU activation, and another linear layer with `out_features=10`. Pass a new random input image through this sequential model to get the logits.

<details><summary><b>Solution</b></summary> <pre> 

```python
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
```

</pre> </details>

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.

## Exercise 6 - Apply Softmax
Apply the softmax function to the logits to get the predicted probabilities.



<details><summary><b>Solution</b></summary> <pre> 

```python
softmax = nn.Softmax(dim=1) 
pred_probab = softmax(logits) 
```

</pre> </details>

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.

## Exercise 7 - Print Model Structure and Parameters
Print the structure of the model and the size and values of its parameters.

<details><summary><b>Solution</b></summary> <pre> 


```python
print("Model structure: ", model, "\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")
```

</pre>
</details>